In [1]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.9 MB/s eta 0:00:00


In [2]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. The model may run slowly.")

GPU available: True
GPU: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

Loading model...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [4]:
messages = [
    {
        "role": "user",
        "content": "What is an AI agent? Explain in one sentence."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Model response:")
print(answer)

Model response:
An AI agent is an artificial intelligence system designed to perceive its environment and take actions that maximize its chance of success according to some goal.


In [18]:
from google.colab import files

print("Upload one or more meeting transcripts (.txt)")

uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} file(s).")

for filename in uploaded.keys():
    print("✓", filename)

Upload one or more meeting transcripts (.txt)


Saving Weekly Meeting HR.txt to Weekly Meeting HR (1).txt
Saving Weekly Meeting Marketing Campaign.txt to Weekly Meeting Marketing Campaign (1).txt
Saving Weekly Meeting Product Development.txt to Weekly Meeting Product Development (1).txt
Saving Weekly Meeting Project Phoenix.txt to Weekly Meeting Project Phoenix (1).txt

Uploaded 4 file(s).
✓ Weekly Meeting HR (1).txt
✓ Weekly Meeting Marketing Campaign (1).txt
✓ Weekly Meeting Product Development (1).txt
✓ Weekly Meeting Project Phoenix (1).txt


In [22]:
import os
import json

os.makedirs("outputs", exist_ok=True)

print("Output folder ready.")

Output folder ready.


In [23]:
for filename in uploaded.keys():

    print("\n" + "=" * 70)
    print("PROCESSING:", filename)
    print("=" * 70)

    # -----------------------------------
    # Read meeting transcript
    # -----------------------------------

    with open(filename, "r", encoding="utf-8") as f:
        meeting_transcript = f.read()

    # -----------------------------------
    # Create prompt
    # -----------------------------------

    prompt = f"""
You are a professional meeting analysis assistant.

Analyze the meeting transcript below.

Extract:

1. A concise meeting summary.
2. Important decisions.
3. EVERY explicit action item.

Return ONLY valid JSON.

Use exactly this structure:

{{
  "summary": "A concise summary of the meeting",
  "decisions": [
    "Decision 1"
  ],
  "action_items": [
    {{
      "task": "Task description",
      "owner": "Person responsible",
      "due_date": "Due date, dependency, or Not specified"
    }}
  ]
}}

RULES:

- Extract EVERY explicit action or commitment.
- Only assign an owner when the transcript explicitly identifies the person.
- If the owner is unknown, use "Unassigned".
- Only use deadlines explicitly stated in the transcript.
- Do not invent or infer deadlines.
- If no deadline is stated, use "Not specified".
- Preserve dependencies such as "after the backend is ready".
- Include conditional actions such as "if UI issues are found".
- Only identify decisions that are explicitly agreed, approved, finalized, or decided.
- Do not invent information.
- Do not add explanations outside the JSON.
- Return ONLY valid JSON.

MEETING TRANSCRIPT:

{meeting_transcript}
"""

    # -----------------------------------
    # Prepare model input
    # -----------------------------------

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # -----------------------------------
    # Generate response
    # -----------------------------------

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False
    )

    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # -----------------------------------
    # Clean model response
    # -----------------------------------

    result = result.strip()

    if "```json" in result:
        result = result.split("```json", 1)[1].split("```", 1)[0].strip()

    elif "```" in result:
        result = result.split("```", 1)[1].split("```", 1)[0].strip()

    # -----------------------------------
    # Convert response to JSON
    # -----------------------------------

    try:

        data = json.loads(result)

        print("✓ JSON valid")

        # -----------------------------------
        # Save JSON output
        # -----------------------------------

        base_name = os.path.splitext(filename)[0]

        output_file = f"outputs/{base_name}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print("✓ Saved:", output_file)

        # -----------------------------------
        # Display summary
        # -----------------------------------

        print("\nSUMMARY:")
        print(data["summary"])

        # -----------------------------------
        # Display decisions
        # -----------------------------------

        print("\nDECISIONS:")

        if data["decisions"]:

            for i, decision in enumerate(data["decisions"], 1):
                print(f"{i}. {decision}")

        else:

            print("No explicit decisions identified.")

        # -----------------------------------
        # Display action items
        # -----------------------------------

        print("\nACTION ITEMS:")

        for i, item in enumerate(data["action_items"], 1):

            print(
                f"{i}. {item['task']} | "
                f"Owner: {item['owner']} | "
                f"Due: {item['due_date']}"
            )

    except json.JSONDecodeError:

        print("✗ ERROR: Model returned invalid JSON.")

        print("\nRAW MODEL OUTPUT:")
        print(result)

print("\n" + "=" * 70)
print("ALL FILES PROCESSED")
print("=" * 70)


PROCESSING: Weekly Meeting HR (1).txt
✓ JSON valid
✓ Saved: outputs/Weekly Meeting HR (1).json

SUMMARY:
The team discussed and assigned tasks related to the employee onboarding process and the employee handbook. Priya will update the new employee onboarding checklist by Thursday. Rahul will review the checklist after it is completed. Ananya will work on updating the employee handbook this week. Rahul will schedule a review meeting with HR after the checklist is finalized.

DECISIONS:
1. Agreed to use the updated onboarding process for all new employees.

ACTION ITEMS:
1. Update the new employee onboarding checklist | Owner: Priya | Due: Thursday
2. Review the updated onboarding checklist | Owner: Rahul | Due: Not specified
3. Work on updating the employee handbook | Owner: Ananya | Due: This week
4. Schedule a review meeting with HR after the checklist is finalized | Owner: Rahul | Due: Not specified

PROCESSING: Weekly Meeting Marketing Campaign (1).txt
✓ JSON valid
✓ Saved: outputs

In [24]:
import os

print("JSON files:")

for file in os.listdir("outputs"):
    if file.endswith(".json"):
        print("✓", file)

JSON files:
✓ Weekly Meeting HR (1).json
✓ Weekly Meeting Project Phoenix (1).json
✓ Weekly Meeting Product Development (1).json
✓ Weekly Meeting Marketing Campaign (1).json
